# Qwen3 Dense Embedding Experiment Pipeline (Kaggle)

This notebook executes and evaluates dense embedding retrieval experiments on Vietnamese news corpus chunks using the **Qwen/Qwen3-Embedding-0.6B** model.
It compares retrieval performance across four chunking strategies:
1. `token`: Token-based chunk splitter.
2. `structured`: Structure-aware block splitter.
3. `llamaindex`: Sentence-splitter by LlamaIndex.
4. `langchain_recursive`: Recursive character splitter by LangChain.

This notebook starts directly from pre-chunked JSONL files and does not perform chunking. It outputs FAISS indices, embeddings, retrieval lists, and evaluates standard information retrieval metrics.


## 1. Install dependencies
Installs required packages in the environment (e.g., Kaggle, Colab, or local virtual environments).


In [ ]:
# 1. Install dependencies
import sys
import subprocess

def install_packages():
    """Installs required third-party libraries for the embedding experiment."""
    packages = [
        "sentence-transformers>=2.7.0",
        "transformers>=4.51.0",
        "accelerate",
        "faiss-cpu",
        "orjson",
        "pandas",
        "numpy",
        "tqdm",
        "tabulate"
    ]
    print("Installing packages: ", ", ".join(packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
    print("Installation completed successfully.")

install_packages()


## 2. Imports
Imports all standard and third-party modules needed for path resolving, data serialization, embedding encoding, FAISS indexing, dense retrieval, and metrics evaluation.


In [ ]:
# 2. Imports
import gc
import json
import logging
import math
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import torch
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import orjson
import pandas as pd
from tqdm.auto import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("DenseEmbeddingPipeline")
logger.info("Libraries imported and logger initialized.")


## 3. Configuration
Contains user-adjustable variables including data path directories, model names, batch sizes, and hardware parameters.
By default, the path configurations support both local workspace folders and Kaggle dataset input directories.


In [ ]:
# 3. Configuration
@dataclass
class PipelineConfig:
    # Model configuration
    model_name: str = "Qwen/Qwen3-Embedding-0.6B"
    batch_size: int = 16
    max_seq_length: int = 512
    attn_implementation: str = "eager"
    normalize_embeddings: bool = True
    
    # Input/Output paths
    # Note: These paths fall back to standard Kaggle structures if local directories are missing
    input_dir: str = "src/embed/output/chunks"  # Kaggle fallback is handled dynamically
    qa_path: str = "Dataset/QA_Claude/QA_output.jsonl"
    output_dir: str = "src/embed/output/dense"
    
    # Strategies to evaluate
    strategies: List[str] = None
    
    # Evaluation configuration
    top_k: int = 10
    
    def __post_init__(self):
        if self.strategies is None:
            self.strategies = ["token", "structured", "llamaindex", "langchain_recursive"]

# Initialize default configuration
config = PipelineConfig()
logger.info(f"Pipeline Config initialized: {config}")


## 4. Load Model
Initializes the `SentenceTransformer` model using remote code trust, left-tokenizer padding, eager attention execution, and bounds context length to 512 tokens.


In [ ]:
# 4. Load Model
def load_embedding_model(config: PipelineConfig) -> SentenceTransformer:
    """Loads the sentence-transformer embedding model with the specified configs."""
    logger.info(f"Loading embedding model: {config.model_name}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model_kwargs = {
        "trust_remote_code": True,
        "attn_implementation": config.attn_implementation
    }
    tokenizer_kwargs = {
        "padding_side": "left"
    }
    
    model = SentenceTransformer(
        config.model_name,
        trust_remote_code=True,
        model_kwargs=model_kwargs,
        tokenizer_kwargs=tokenizer_kwargs,
        device=device
    )
    
    # Apply special model-specific requirements
    model.max_seq_length = config.max_seq_length
    if hasattr(model, "tokenizer") and model.tokenizer is not None:
        model.tokenizer.padding_side = "left"
        
    logger.info(f"Model loaded successfully on device: {device}")
    logger.info(f"Tokenizer padding_side: {getattr(model.tokenizer, 'padding_side', 'unknown')}")
    logger.info(f"Max sequence length set to: {model.max_seq_length}")
    
    return model


## 5. Utility Functions
Defines reusable methods for formatting documents and queries, reading and writing files efficiently using `orjson`, and measuring directory file sizes.


In [ ]:
# 5. Utility Functions
def format_document(title: str, category: str, text: str) -> str:
    """Formats the document text with Title, Category, and Content sections."""
    return f"Title: {title}\n\nCategory: {category}\n\nContent:\n{text}"

def format_query(question: str) -> str:
    """Formats the query question with Qwen3 search prompt instructions."""
    return f"Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:\n{question}"

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Reads a JSONL file using fast orjson deserializer."""
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(orjson.loads(line))
    return data

def write_jsonl(path: Path, data: List[Dict[str, Any]]) -> None:
    """Writes a list of dictionaries to a JSONL file using orjson."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        for item in data:
            f.write(orjson.dumps(item))
            f.write(b"\n")

def get_directory_size_mb(path: Path) -> float:
    """Computes the total size of files under the path in Megabytes."""
    if not path.exists():
        return 0.0
    if path.is_file():
        return path.stat().st_size / (1024 * 1024)
    total_size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total_size / (1024 * 1024)


## 6. Load Chunk Dataset
Loads chunk datasets for chunking strategies, supporting multiple common workspace and Kaggle input naming conventions.


In [ ]:
# 6. Load Chunk Dataset
def resolve_chunk_file_path(strategy: str, config: PipelineConfig) -> Path:
    """Resolves path to chunk JSONL file under the input directory, supporting multiple naming variants."""
    input_dir = Path(config.input_dir)
    candidates = [
        input_dir / f"{strategy}.jsonl",
        input_dir / f"vieonline_news_chunks_{strategy}.jsonl",
        Path("src/embed/output/chunks") / f"{strategy}.jsonl",
        Path("src/embed/output/chunks") / f"vieonline_news_chunks_{strategy}.jsonl",
        Path("/input/chunks") / f"{strategy}.jsonl",
        Path("/input/chunks") / f"vieonline_news_chunks_{strategy}.jsonl"
    ]
    for c in candidates:
        if c.exists():
            logger.info(f"Resolved chunk file path for strategy '{strategy}' to: {c}")
            return c
    raise FileNotFoundError(
        f"Could not locate chunk file for strategy '{strategy}' in configuration path '{config.input_dir}' "
        f"or common fallback folders. Tried candidates: {[str(c) for c in candidates]}"
    )

def load_chunk_dataset(strategy: str, config: PipelineConfig) -> List[Dict[str, Any]]:
    """Loads and returns chunk dataset for the specified strategy."""
    path = resolve_chunk_file_path(strategy, config)
    chunks = read_jsonl(path)
    logger.info(f"Loaded {len(chunks)} chunks for strategy '{strategy}'.")
    return chunks


## 7. Generate Embeddings
Generates semantic embeddings for each document formatted text. Uses single-GPU sequential batch encoding with mixed precision to guarantee CUDA stability.


In [ ]:
# 7. Generate Embeddings
def generate_strategy_embeddings(
    chunks: List[Dict[str, Any]], 
    model: SentenceTransformer, 
    config: PipelineConfig
) -> Tuple[np.ndarray, float, float]:
    """Generates embeddings for chunks, formatting them appropriately and tracking performance."""
    logger.info(f"Generating embeddings for {len(chunks)} chunks...")
    
    formatted_texts = []
    for chunk in chunks:
        meta = chunk.get("metadata", {})
        title = meta.get("title") or ""
        category = meta.get("category") or ""
        text = chunk.get("text") or ""
        
        formatted = format_document(title=title, category=category, text=text)
        formatted_texts.append(formatted)
        
    start_time = time.perf_counter()
    device = getattr(model, "device", None) or "cpu"
    is_cuda = "cuda" in str(device)
    
    embeddings_list = []
    batch_size = config.batch_size
    
    with torch.no_grad():
        for i in tqdm(range(0, len(formatted_texts), batch_size), desc="Encoding batches", unit="batch"):
            batch = formatted_texts[i : i + batch_size]
            
            if is_cuda:
                with torch.cuda.amp.autocast():
                    vectors = model.encode(
                        batch,
                        batch_size=batch_size,
                        normalize_embeddings=config.normalize_embeddings,
                        show_progress_bar=False,
                        convert_to_numpy=True
                    )
            else:
                vectors = model.encode(
                    batch,
                    batch_size=batch_size,
                    normalize_embeddings=config.normalize_embeddings,
                    show_progress_bar=False,
                    convert_to_numpy=True
                )
            embeddings_list.append(vectors)
            
    embeddings = np.vstack(embeddings_list).astype(np.float32)
    elapsed_time = time.perf_counter() - start_time
    chunks_per_sec = len(chunks) / elapsed_time if elapsed_time > 0 else 0.0
    
    logger.info(f"Generated embeddings array of shape {embeddings.shape} in {elapsed_time:.2f} seconds ({chunks_per_sec:.2f} chunks/sec).")
    return embeddings, elapsed_time, chunks_per_sec


## 8. Save Embeddings
Writes all generated embeddings, manifests, stats, metadata rows, and first-5 debug samples to disk.


In [ ]:
# 8. Save Embeddings
def compute_length_stats(values: List[int]) -> Dict[str, Any]:
    """Helper to return min, max, avg, and sum of a sequence of integers."""
    if not values:
        return {"min": 0, "max": 0, "avg": 0.0, "sum": 0}
    return {
        "min": int(np.min(values)),
        "max": int(np.max(values)),
        "avg": float(np.mean(values)),
        "sum": int(np.sum(values))
    }

def save_strategy_outputs(
    strategy: str,
    chunks: List[Dict[str, Any]],
    embeddings: np.ndarray,
    elapsed_time: float,
    chunks_per_sec: float,
    config: PipelineConfig
) -> Dict[str, Path]:
    """Saves embedding output files, metadata, manifest, stats, and debug samples."""
    strategy_dir = Path(config.output_dir) / strategy
    strategy_dir.mkdir(parents=True, exist_ok=True)
    
    embeddings_path = strategy_dir / "embeddings.npy"
    metadata_path = strategy_dir / "metadata.jsonl"
    manifest_path = strategy_dir / "manifest.json"
    stats_path = strategy_dir / "embedding_stats.json"
    debug_samples_path = strategy_dir / "debug_samples.jsonl"
    
    np.save(embeddings_path, embeddings)
    write_jsonl(metadata_path, chunks)
    
    manifest = {
        "strategy": strategy,
        "model_name": config.model_name,
        "embedding_dimension": int(embeddings.shape[1]),
        "normalize_embeddings": config.normalize_embeddings,
        "batch_size": config.batch_size,
        "chunk_count": len(chunks),
        "created_at": datetime.now(timezone.utc).isoformat()
    }
    with open(manifest_path, "wb") as f:
        f.write(orjson.dumps(manifest, option=orjson.OPT_INDENT_2))
        
    debug_samples = []
    for i in range(min(5, len(chunks))):
        c = chunks[i]
        meta = c.get("metadata", {})
        formatted_txt = format_document(
            title=meta.get("title") or "",
            category=meta.get("category") or "",
            text=c.get("text") or ""
        )
        debug_samples.append({
            "chunk_id": c.get("chunk_id"),
            "article_id": c.get("article_id"),
            "formatted_text": formatted_txt,
            "embedding_input_preview": formatted_txt[:500] + ("..." if len(formatted_txt) > 500 else "")
        })
    write_jsonl(debug_samples_path, debug_samples)
    
    token_counts = [int(c.get("metadata", {}).get("token_count") or len(str(c.get("text", "")).split())) for c in chunks]
    char_counts = [len(format_document(c.get("metadata", {}).get("title") or "", c.get("metadata", {}).get("category") or "", c.get("text") or "")) for c in chunks]
    
    stats = {
        "embedding_time_seconds": float(elapsed_time),
        "chunks_per_second": float(chunks_per_sec),
        "embedding_dimension": int(embeddings.shape[1]),
        "index_size_mb": 0.0,
        "token_count_stats": compute_length_stats(token_counts),
        "character_count_stats": compute_length_stats(char_counts)
    }
    with open(stats_path, "wb") as f:
        f.write(orjson.dumps(stats, option=orjson.OPT_INDENT_2))
        
    logger.info(f"Saved strategy outputs successfully at: {strategy_dir}")
    return {
        "embeddings": embeddings_path,
        "metadata": metadata_path,
        "manifest": manifest_path,
        "stats": stats_path,
        "debug_samples": debug_samples_path
    }


## 9. Build FAISS Index
Loads standard normalized embeddings into a flat inner product FAISS index. Saves the index and updates index size measurements.


In [ ]:
# 9. Build FAISS Index
def build_faiss_index(
    strategy: str,
    embeddings: np.ndarray,
    config: PipelineConfig
) -> Path:
    """Builds a FAISS IndexFlatIP index and updates strategy embedding_stats."""
    logger.info(f"Building FAISS IndexFlatIP index for strategy '{strategy}'...")
    
    strategy_dir = Path(config.output_dir) / strategy
    index_path = strategy_dir / "index.faiss"
    stats_path = strategy_dir / "embedding_stats.json"
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings.astype(np.float32))
    
    faiss.write_index(index, str(index_path))
    
    index_size_mb = get_directory_size_mb(index_path)
    logger.info(f"Saved FAISS index to {index_path} (Size: {index_size_mb:.4f} MB)")
    
    if stats_path.exists():
        with open(stats_path, "rb") as f:
            stats = orjson.loads(f.read())
        stats["index_size_mb"] = float(index_size_mb)
        with open(stats_path, "wb") as f:
            f.write(orjson.dumps(stats, option=orjson.OPT_INDENT_2))
            
    return index_path


## 10. Dense Retrieval
Parses the ground truth query and relevance set (qrels), runs vector search for all queries, and logs retrieve latency.


In [ ]:
# 10. Dense Retrieval
def resolve_qa_file_path(config: PipelineConfig) -> Path:
    """Resolves QA dataset path, checking configuration and common fallback directories."""
    qa_path = Path(config.qa_path)
    candidates = [
        qa_path,
        Path("Dataset/QA_Claude/QA_output.jsonl"),
        Path("Dataset/QA_Claude/QA_output.csv"),
        Path("/input/QA_output.jsonl"),
        Path("/input/QA_output.csv"),
        Path("src/embed/output/data/vieonline_news_clean.jsonl")
    ]
    for c in candidates:
        if c.exists():
            logger.info(f"Resolved QA dataset path to: {c}")
            return c
    raise FileNotFoundError(
        f"Could not locate QA dataset file at '{config.qa_path}' or fallback locations: "
        f"{[str(c) for c in candidates]}"
    )

def load_qa_dataset(config: PipelineConfig) -> Tuple[List[Dict[str, Any]], Dict[str, Set[str]]]:
    """Loads QA dataset, filtering out invalid questions and returning queries and qrels."""
    path = resolve_qa_file_path(config)
    queries = []
    qrels = {}
    
    if path.suffix == ".csv":
        df = pd.read_csv(path)
        if "is_possible" in df.columns:
            df = df[df["is_possible"].astype(str).str.lower().isin(["true", "1", "yes"])]
        df = df.dropna(subset=["question"])
        for _, row in df.iterrows():
            q_id = str(row.get("id") or row.name)
            question = str(row["question"])
            
            rel_ids = set()
            for col in ["source_article_ids", "article_id"]:
                if col in row and pd.notna(row[col]):
                    val = str(row[col]).strip()
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, list):
                            rel_ids.update(str(x) for x in parsed)
                        else:
                            rel_ids.add(str(parsed))
                    except Exception:
                        for p in val.replace(";", ",").replace("|", ",").split(","):
                            p_clean = p.strip().strip("[]'\"")
                            if p_clean and p_clean.lower() not in {"nan", "none", ""}:
                                rel_ids.add(p_clean)
            
            queries.append({"query_id": q_id, "question": question})
            qrels[q_id] = rel_ids
    else:
        rows = read_jsonl(path)
        for idx, row in enumerate(rows):
            if "is_possible" in row and not row["is_possible"]:
                continue
            q_id = str(row.get("id") or row.get("query_id") or idx)
            question = row.get("question")
            if not question:
                continue
            
            rel_ids = set()
            for key in ["source_article_ids", "article_id"]:
                if key in row and row[key] is not None:
                    val = row[key]
                    if isinstance(val, list):
                        rel_ids.update(str(x) for x in val)
                    else:
                        val_str = str(val).strip()
                        try:
                            parsed = json.loads(val_str)
                            if isinstance(parsed, list):
                                rel_ids.update(str(x) for x in parsed)
                            else:
                                rel_ids.add(str(parsed))
                        except Exception:
                            for p in val_str.replace(";", ",").replace("|", ",").split(","):
                                p_clean = p.strip().strip("[]'\"")
                                if p_clean and p_clean.lower() not in {"nan", "none", ""}:
                                    rel_ids.add(p_clean)
            
            queries.append({"query_id": q_id, "question": question})
            qrels[q_id] = rel_ids
            
    logger.info(f"Loaded {len(queries)} queries and qrels from {path}")
    return queries, qrels

def run_dense_retrieval(
    strategy: str,
    queries: List[Dict[str, Any]],
    model: SentenceTransformer,
    config: PipelineConfig
) -> Tuple[List[Dict[str, Any]], List[float]]:
    """Runs dense retrieval for all queries, using FAISS index to retrieve Top-K chunks."""
    strategy_dir = Path(config.output_dir) / strategy
    index_path = strategy_dir / "index.faiss"
    metadata_path = strategy_dir / "metadata.jsonl"
    results_path = strategy_dir / "retrieval_results.jsonl"
    
    if not index_path.exists():
        raise FileNotFoundError(f"FAISS index not found for strategy '{strategy}' at: {index_path}")
    index = faiss.read_index(str(index_path))
    metadata = read_jsonl(metadata_path)
    
    retrieval_results = []
    latencies_ms = []
    
    logger.info(f"Running dense retrieval for {len(queries)} queries on strategy '{strategy}'...")
    
    for q in tqdm(queries, desc=f"Retrieving '{strategy}'", unit="query"):
        query_id = q["query_id"]
        question = q["question"]
        
        formatted_q = format_query(question)
        start_time = time.perf_counter()
        
        query_vector = model.encode(
            [formatted_q], 
            normalize_embeddings=config.normalize_embeddings, 
            show_progress_bar=False,
            convert_to_numpy=True
        ).astype(np.float32)
        
        scores, indices = index.search(query_vector, config.top_k)
        latency_ms = (time.perf_counter() - start_time) * 1000.0
        latencies_ms.append(latency_ms)
        
        retrieved_chunks = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
            if idx < 0 or idx >= len(metadata):
                continue
            chunk_info = metadata[idx]
            retrieved_chunks.append({
                "chunk_id": chunk_info.get("chunk_id"),
                "article_id": str(chunk_info.get("article_id")),
                "score": float(score),
                "rank": rank,
                "metadata": {
                    "title": chunk_info.get("metadata", {}).get("title"),
                    "category": chunk_info.get("metadata", {}).get("category"),
                    "token_count": chunk_info.get("metadata", {}).get("token_count")
                }
            })
            
        retrieval_results.append({
            "query_id": query_id,
            "query_text": question,
            "retrieved": retrieved_chunks
        })
        
    write_jsonl(results_path, retrieval_results)
    logger.info(f"Saved retrieval results to: {results_path}")
    return retrieval_results, latencies_ms


## 11. Evaluation
Implements evaluation metrics: Recall@5, Recall@10, MRR@10, Hit@1, Hit@5, nDCG@10.


In [ ]:
# 11. Evaluation
def dcg_at_k(relevances: List[int], k: int) -> float:
    """Calculates Discounted Cumulative Gain at K for binary relevance."""
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances[:k]))

def ndcg_at_k(retrieved_article_ids: List[str], relevant_article_ids: Set[str], k: int) -> float:
    """Calculates Normalized Discounted Cumulative Gain at K for binary relevance."""
    relevances = [1 if doc_id in relevant_article_ids else 0 for doc_id in retrieved_article_ids[:k]]
    actual_dcg = dcg_at_k(relevances, k)
    
    num_relevant = len(relevant_article_ids)
    ideal_relevances = [1] * min(num_relevant, k)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    
    if ideal_dcg == 0.0:
        return 0.0
    return actual_dcg / ideal_dcg

def evaluate_retrieval(
    retrieval_results: List[Dict[str, Any]],
    qrels: Dict[str, Set[str]]
) -> Dict[str, float]:
    """Computes Recall, MRR, Hit Rate, and nDCG metrics across all queries."""
    recall_5_list = []
    recall_10_list = []
    mrr_10_list = []
    hit_1_list = []
    hit_5_list = []
    ndcg_10_list = []
    
    for res in retrieval_results:
        query_id = res["query_id"]
        relevant_article_ids = qrels.get(query_id, set())
        if not relevant_article_ids:
            continue
            
        retrieved = res["retrieved"]
        retrieved_article_ids = [item["article_id"] for item in retrieved]
        
        retrieved_5 = set(retrieved_article_ids[:5])
        
        recall_5 = len(retrieved_5 & relevant_article_ids) / len(relevant_article_ids)
        recall_10 = len(set(retrieved_article_ids[:10]) & relevant_article_ids) / len(relevant_article_ids)
        
        recall_5_list.append(recall_5)
        recall_10_list.append(recall_10)
        
        hit_1 = 1.0 if retrieved_article_ids[:1] and retrieved_article_ids[0] in relevant_article_ids else 0.0
        hit_5 = 1.0 if retrieved_5 & relevant_article_ids else 0.0
        
        hit_1_list.append(hit_1)
        hit_5_list.append(hit_5)
        
        mrr = 0.0
        for rank, doc_id in enumerate(retrieved_article_ids[:10], start=1):
            if doc_id in relevant_article_ids:
                mrr = 1.0 / rank
                break
        mrr_10_list.append(mrr)
        
        ndcg_10 = ndcg_at_k(retrieved_article_ids, relevant_article_ids, 10)
        ndcg_10_list.append(ndcg_10)
        
    metrics = {
        "Recall@5": float(np.mean(recall_5_list)) if recall_5_list else 0.0,
        "Recall@10": float(np.mean(recall_10_list)) if recall_10_list else 0.0,
        "MRR@10": float(np.mean(mrr_10_list)) if mrr_10_list else 0.0,
        "Hit@1": float(np.mean(hit_1_list)) if hit_1_list else 0.0,
        "Hit@5": float(np.mean(hit_5_list)) if hit_5_list else 0.0,
        "nDCG@10": float(np.mean(ndcg_10_list)) if ndcg_10_list else 0.0
    }
    
    logger.info(f"Evaluation metrics computed for {len(recall_5_list)} queries.")
    return metrics


## 12. Generate Statistics
Gathers all model stats (encoding speed, dimension size, latencies, index size) and saves to `metrics.json`.


In [ ]:
# 12. Generate Statistics
def generate_and_save_metrics(
    strategy: str,
    eval_metrics: Dict[str, float],
    latencies_ms: List[float],
    config: PipelineConfig
) -> Dict[str, Any]:
    """Generates the final metrics dictionary containing retrieval performance and resource usage, then saves to metrics.json."""
    strategy_dir = Path(config.output_dir) / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    metrics_path = strategy_dir / "metrics.json"
    
    embedding_stats = {}
    if stats_path.exists():
        with open(stats_path, "rb") as f:
            embedding_stats = orjson.loads(f.read())
            
    avg_latency = float(np.mean(latencies_ms)) if latencies_ms else 0.0
    p95_latency = float(np.percentile(latencies_ms, 95)) if latencies_ms else 0.0
    
    final_metrics = {
        "Recall@5": eval_metrics.get("Recall@5", 0.0),
        "Recall@10": eval_metrics.get("Recall@10", 0.0),
        "MRR@10": eval_metrics.get("MRR@10", 0.0),
        "Hit@1": eval_metrics.get("Hit@1", 0.0),
        "Hit@5": eval_metrics.get("Hit@5", 0.0),
        "nDCG@10": eval_metrics.get("nDCG@10", 0.0),
        "embedding_time_seconds": embedding_stats.get("embedding_time_seconds", 0.0),
        "chunks_per_second": embedding_stats.get("chunks_per_second", 0.0),
        "embedding_dimension": embedding_stats.get("embedding_dimension", 0),
        "index_size_mb": embedding_stats.get("index_size_mb", 0.0),
        "query_latency_ms_avg": avg_latency,
        "query_latency_ms_p95": p95_latency
    }
    
    with open(metrics_path, "wb") as f:
        f.write(orjson.dumps(final_metrics, option=orjson.OPT_INDENT_2))
        
    logger.info(f"Saved complete metrics.json for strategy '{strategy}' to: {metrics_path}")
    return final_metrics


## 13. Leaderboard Summary
Compiles and prints a comparative markdown table comparing nDCG@10, Recall@10, MRR@10, Index Size, Latency, and Encoding Speed.


In [ ]:
# 13. Leaderboard Summary
def display_leaderboard(config: PipelineConfig) -> None:
    """Aggregates metrics for all completed strategies and prints a comparing leaderboard table."""
    logger.info("Generating Leaderboard Summary...")
    results = []
    
    for strategy in config.strategies:
        metrics_path = Path(config.output_dir) / strategy / "metrics.json"
        if metrics_path.exists():
            with open(metrics_path, "rb") as f:
                m = orjson.loads(f.read())
            results.append({
                "Strategy": strategy,
                "nDCG@10": float(m.get("nDCG@10", 0.0)),
                "Recall@10": float(m.get("Recall@10", 0.0)),
                "MRR@10": float(m.get("MRR@10", 0.0)),
                "Hit@1": float(m.get("Hit@1", 0.0)),
                "Latency Avg (ms)": float(m.get("query_latency_ms_avg", 0.0)),
                "Latency P95 (ms)": float(m.get("query_latency_ms_p95", 0.0)),
                "Index Size (MB)": float(m.get("index_size_mb", 0.0)),
                "Chunks/Sec": float(m.get("chunks_per_second", 0.0))
            })
            
    if not results:
        logger.warning("No metrics.json files found. Run the embedding generation first.")
        return
        
    # Sort leaderboard by nDCG@10 descending, then Recall@10 descending, then latency ascending
    results = sorted(results, key=lambda x: (-x["nDCG@10"], -x["Recall@10"], x["Latency Avg (ms)"]))
    
    headers = [
        "Strategy", "nDCG@10", "Recall@10", "MRR@10", "Hit@1", 
        "Latency Avg (ms)", "Latency P95 (ms)", "Index Size (MB)", "Chunks/Sec"
    ]
    
    print("\n" + "=" * 35 + " DENSE EMBEDDING RETRIEVAL LEADERBOARD " + "=" * 35)
    
    # Print headers
    header_row = " | ".join(headers)
    divider_row = " | ".join(["---"] * len(headers))
    print(f"| {header_row} |")
    print(f"| {divider_row} |")
    
    # Print rows
    for r in results:
        row_str = (
            f"| {r['Strategy']} | "
            f"{r['nDCG@10']:.4f} | "
            f"{r['Recall@10']:.4f} | "
            f"{r['MRR@10']:.4f} | "
            f"{r['Hit@1']:.4f} | "
            f"{r['Latency Avg (ms)']:.2f} | "
            f"{r['Latency P95 (ms)']:.2f} | "
            f"{r['Index Size (MB)']:.4f} | "
            f"{r['Chunks/Sec']:.2f} |"
        )
        print(row_str)
        
    print("=" * 110 + "\n")


## 14. Error Handling
Defines wrappers `run_strategy` and `run_all` that catch exceptions, allow running remaining strategies if one fails, and support resume/skip options.


In [ ]:
# 14. Error Handling
def run_strategy(
    strategy: str,
    model: SentenceTransformer,
    queries: List[Dict[str, Any]],
    qrels: Dict[str, Set[str]],
    config: PipelineConfig
) -> None:
    """Executes pipeline steps for a single strategy: Loading dataset, generating/loading embeddings, building index, retrieval, and evaluation."""
    logger.info(f"======================================== Starting strategy: {strategy} ========================================")
    strategy_dir = Path(config.output_dir) / strategy
    strategy_dir.mkdir(parents=True, exist_ok=True)
    
    embeddings_path = strategy_dir / "embeddings.npy"
    index_path = strategy_dir / "index.faiss"
    metrics_path = strategy_dir / "metrics.json"
    
    # 1. Skip entire strategy if metrics.json already exists (Resume support)
    if metrics_path.exists():
        logger.info(f"Strategy '{strategy}' metrics.json already exists. Skipping entirely.")
        return
        
    # Load chunks (needed for saving outputs, metadata)
    chunks = load_chunk_dataset(strategy, config)
    
    # 2. Check if embeddings.npy exists to skip embedding step (Resume/skip support)
    if embeddings_path.exists():
        logger.info(f"Embeddings file for '{strategy}' already exists at {embeddings_path}. Loading existing embeddings...")
        embeddings = np.load(embeddings_path)
        elapsed_time = 0.0
        chunks_per_sec = 0.0
        # Re-save metadata/manifest/stats in case they were deleted/corrupted
        save_strategy_outputs(strategy, chunks, embeddings, elapsed_time, chunks_per_sec, config)
    else:
        embeddings, elapsed_time, chunks_per_sec = generate_strategy_embeddings(chunks, model, config)
        save_strategy_outputs(strategy, chunks, embeddings, elapsed_time, chunks_per_sec, config)
        
    # 3. Build FAISS index if missing
    if not index_path.exists():
        build_faiss_index(strategy, embeddings, config)
    else:
        logger.info(f"FAISS index for '{strategy}' already exists at {index_path}. Skipping build.")
        
    # 4. Dense Retrieval
    retrieval_results, latencies_ms = run_dense_retrieval(strategy, queries, model, config)
    
    # 5. Evaluation
    eval_metrics = evaluate_retrieval(retrieval_results, qrels)
    
    # 6. Save Stats & Metrics
    generate_and_save_metrics(strategy, eval_metrics, latencies_ms, config)
    logger.info(f"Completed strategy successfully: {strategy}")

def run_all(config: PipelineConfig = None) -> None:
    """Runs dense embedding pipeline for all configured strategies, handling exceptions to proceed with remaining strategies."""
    if config is None:
        config = PipelineConfig()
        
    logger.info(f"Starting Dense Embedding Experiments with {len(config.strategies)} strategies.")
    
    # Load QA dataset first
    try:
        queries, qrels = load_qa_dataset(config)
    except Exception as e:
        logger.critical(f"Failed to load QA dataset. Pipeline cannot continue. Error: {e}", exc_info=True)
        return
        
    # Load model once and share it
    try:
        model = load_embedding_model(config)
    except Exception as e:
        logger.critical(f"Failed to load embedding model {config.model_name}. Error: {e}", exc_info=True)
        return
        
    for strategy in config.strategies:
        try:
            run_strategy(strategy, model, queries, qrels, config)
        except Exception as e:
            logger.error(f"Strategy '{strategy}' failed during execution! Error: {e}", exc_info=True)
            logger.info("Continuing with other strategies as requested...")
            
        # Free GPU memory before next strategy
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    # Generate final leaderboard
    try:
        display_leaderboard(config)
    except Exception as e:
        logger.error(f"Failed to display leaderboard: {e}", exc_info=True)
        
    logger.info("Embedding experiments run completed.")

# Execute pipeline if run in Jupyter/Python environment
if __name__ == "__main__" or "IPython" in sys.modules:
    run_all(config)
